# Real-Time SDR Monitoring Dashboard

## Live Trade Flow Monitoring for US Interest Rate Swaps

This notebook provides real-time monitoring capabilities:

1. **Live Trade Feed** - Latest trades as they hit the tape
2. **Volume Tracker** - Real-time volume accumulation
3. **Large Trade Alerts** - Identify significant trades
4. **Rate Monitor** - Track rate levels by tenor
5. **Activity Dashboard** - Current session statistics
6. **Comparison to Historical** - Context vs recent history

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
from IPython.display import display, HTML, clear_output
import time

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder

cache_path = r"/tmp/sdr_cache"
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=False)

## Configuration

In [ ]:
# Configuration parameters
class MonitorConfig:
    # Thresholds for alerts
    LARGE_TRADE_THRESHOLD = 100_000_000  # $100M
    BLOCK_TRADE_HIGHLIGHT = True
    
    # Monitoring settings
    LOOKBACK_MINUTES = 60  # How far back to look for recent trades
    REFRESH_SECONDS = 30  # Auto-refresh interval
    
    # Display settings
    MAX_RECENT_TRADES = 50  # Number of recent trades to show
    
    # Filter settings
    CURRENCY = "USD"
    INDEX = "SOFR"

config = MonitorConfig()

## Data Processing Functions

In [ ]:
def fetch_recent_trades(lookback_minutes: int = 60) -> pd.DataFrame:
    """
    Fetch recent SDR trades.
    """
    now = datetime.datetime.now(NY_tz)
    start = now - datetime.timedelta(minutes=lookback_minutes)
    
    df = sdr.grab_sdr_trades(
        start_timestamp=start,
        end_timestamp=now,
        agency="CFTC",
        asset_class="RATES"
    )
    
    return df

def process_for_display(df: pd.DataFrame) -> pd.DataFrame:
    """
    Process trades for dashboard display.
    """
    if df.empty:
        return df
    
    df = df.copy()
    
    # Filter to new trades
    df = df[df['Action type'] == 'NEWT'].copy()
    
    # Filter USD
    df = df[df['Notional currency-Leg 1'] == config.CURRENCY].copy()
    
    # Filter SOFR
    sofr_mask = df['UPI Underlier Name'].str.contains(config.INDEX, case=False, na=False)
    df = df[sofr_mask].copy()
    
    # Parse timestamps
    df['Event timestamp'] = pd.to_datetime(df['Event timestamp'], utc=True)
    df['Event_Time_NY'] = df['Event timestamp'].dt.tz_convert('America/New_York')
    
    # Parse notional
    df['Notional amount-Leg 1'] = df['Notional amount-Leg 1'].astype(str).str.replace(',', '')
    df['Notional amount-Leg 1'] = pd.to_numeric(df['Notional amount-Leg 1'], errors='coerce')
    
    # Parse fixed rate
    df['Fixed rate-Leg 1'] = pd.to_numeric(df['Fixed rate-Leg 1'], errors='coerce')
    df['Rate_Pct'] = df['Fixed rate-Leg 1'] * 100
    
    # Calculate tenor
    df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
    df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], errors='coerce')
    df['Tenor_Days'] = (df['Expiration Date'] - df['Effective Date']).dt.days
    df['Tenor_Years'] = df['Tenor_Days'] / 365.25
    
    # Tenor label
    def tenor_label(years):
        if pd.isna(years) or years <= 0:
            return 'N/A'
        elif years < 1:
            return f"{int(years*12)}M"
        elif years == int(years):
            return f"{int(years)}Y"
        else:
            return f"{years:.1f}Y"
    
    df['Tenor_Label'] = df['Tenor_Years'].apply(tenor_label)
    
    # Block trade flag
    df['Is_Block'] = df['Block trade election indicator'].astype(str).str.upper() == 'TRUE'
    
    # Large trade flag
    df['Is_Large'] = df['Notional amount-Leg 1'] >= config.LARGE_TRADE_THRESHOLD
    
    # Product type
    def product_type(fisn):
        fisn = str(fisn).upper()
        if 'OIS' in fisn:
            return 'OIS'
        elif 'FXD FLT' in fisn:
            return 'Fixed-Float'
        elif 'CALL' in fisn or 'PUT' in fisn:
            return 'Swaption'
        else:
            return 'Other'
    
    df['Product'] = df['UPI FISN'].apply(product_type)
    
    # Sort by time
    df = df.sort_values('Event_Time_NY', ascending=False)
    
    return df

## 1. Current Session Overview

In [ ]:
# Fetch current data
now = datetime.datetime.now(NY_tz)
today_start = NY_tz.localize(datetime.datetime(now.year, now.month, now.day, 6, 0))

print(f"Fetching trades from {today_start.strftime('%H:%M')} to {now.strftime('%H:%M')} NY...")

raw_df = sdr.grab_sdr_trades(
    start_timestamp=today_start,
    end_timestamp=now,
    agency="CFTC",
    asset_class="RATES"
)

df = process_for_display(raw_df)
print(f"Loaded {len(df):,} USD SOFR trades")

In [ ]:
def display_session_stats(df: pd.DataFrame):
    """
    Display session statistics.
    """
    now = datetime.datetime.now(NY_tz)
    
    total_trades = len(df)
    total_notional = df['Notional amount-Leg 1'].sum()
    avg_size = df['Notional amount-Leg 1'].mean()
    block_count = df['Is_Block'].sum()
    large_count = df['Is_Large'].sum()
    
    # Last trade
    if len(df) > 0:
        last_trade_time = df['Event_Time_NY'].max()
        time_since_last = (now - last_trade_time).total_seconds() / 60
    else:
        time_since_last = float('nan')
    
    print("=" * 60)
    print(f"SESSION STATISTICS - {now.strftime('%Y-%m-%d %H:%M:%S')} NY")
    print("=" * 60)
    print(f"\nTotal Trades:         {total_trades:,}")
    print(f"Total Notional:       ${total_notional:,.0f}")
    print(f"Average Trade Size:   ${avg_size:,.0f}")
    print(f"Block Trades:         {block_count:,} ({block_count/total_trades*100:.1f}%)" if total_trades > 0 else "Block Trades:         0")
    print(f"Large Trades (>${config.LARGE_TRADE_THRESHOLD/1e6:.0f}M): {large_count:,}")
    print(f"\nTime Since Last Trade: {time_since_last:.1f} minutes" if not np.isnan(time_since_last) else "")

display_session_stats(df)

## 2. Recent Trades Feed

In [ ]:
def format_trade_row(row):
    """
    Format a trade row for display.
    """
    flags = []
    if row['Is_Block']:
        flags.append('BLOCK')
    if row['Is_Large']:
        flags.append('LARGE')
    
    flag_str = ' '.join(flags) if flags else ''
    
    return {
        'Time': row['Event_Time_NY'].strftime('%H:%M:%S'),
        'Product': row['Product'],
        'Tenor': row['Tenor_Label'],
        'Rate': f"{row['Rate_Pct']:.4f}%" if pd.notna(row['Rate_Pct']) else 'N/A',
        'Notional': f"${row['Notional amount-Leg 1']/1e6:.1f}M" if pd.notna(row['Notional amount-Leg 1']) else 'N/A',
        'Flags': flag_str
    }

def display_recent_trades(df: pd.DataFrame, n: int = 20):
    """
    Display recent trades in a formatted table.
    """
    if df.empty:
        print("No trades to display.")
        return
    
    recent = df.head(n)
    formatted = [format_trade_row(row) for _, row in recent.iterrows()]
    display_df = pd.DataFrame(formatted)
    
    print(f"\n{'='*80}")
    print(f"RECENT TRADES (Last {n})")
    print(f"{'='*80}")
    display(display_df)

display_recent_trades(df, n=20)

## 3. Large Trade Alerts

In [ ]:
def display_large_trades(df: pd.DataFrame):
    """
    Display large trades with details.
    """
    large_trades = df[df['Is_Large']].copy()
    
    print(f"\n{'='*80}")
    print(f"LARGE TRADE ALERTS (>{config.LARGE_TRADE_THRESHOLD/1e6:.0f}M)")
    print(f"{'='*80}")
    
    if len(large_trades) == 0:
        print("No large trades in the session.")
        return
    
    for _, row in large_trades.iterrows():
        notional = row['Notional amount-Leg 1']
        print(f"\n  [{row['Event_Time_NY'].strftime('%H:%M:%S')}] {row['Product']} {row['Tenor_Label']}")
        print(f"    Notional: ${notional/1e6:.1f}M")
        print(f"    Rate: {row['Rate_Pct']:.4f}%" if pd.notna(row['Rate_Pct']) else "    Rate: N/A")
        print(f"    Platform: {row.get('Platform identifier', 'N/A')}")
        if row['Is_Block']:
            print(f"    ** BLOCK TRADE **")

display_large_trades(df)

## 4. Rate Monitor by Tenor

In [ ]:
def display_rate_monitor(df: pd.DataFrame):
    """
    Display current rate levels by tenor.
    """
    # Filter to swaps with valid rates
    rate_df = df[(df['Rate_Pct'].notna()) & (df['Rate_Pct'] > 0) & (df['Rate_Pct'] < 15)].copy()
    
    if rate_df.empty:
        print("No valid rate data available.")
        return
    
    # Get benchmark tenor trades
    def get_benchmark(years):
        if pd.isna(years):
            return None
        benchmarks = [1, 2, 3, 5, 7, 10, 15, 20, 30]
        for b in benchmarks:
            if abs(years - b) < 0.1:
                return f"{b}Y"
        return None
    
    rate_df['Benchmark'] = rate_df['Tenor_Years'].apply(get_benchmark)
    benchmark_df = rate_df[rate_df['Benchmark'].notna()].copy()
    
    if len(benchmark_df) == 0:
        print("No benchmark tenor trades found.")
        return
    
    # Summary by benchmark
    rate_summary = benchmark_df.groupby('Benchmark').agg({
        'Rate_Pct': ['last', 'mean', 'min', 'max', 'count']
    }).round(4)
    rate_summary.columns = ['Last', 'Average', 'Low', 'High', 'Count']
    
    # Reorder
    tenor_order = ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '15Y', '20Y', '30Y']
    rate_summary = rate_summary.reindex([t for t in tenor_order if t in rate_summary.index])
    
    print(f"\n{'='*70}")
    print("RATE MONITOR - Benchmark Tenors")
    print(f"{'='*70}")
    display(rate_summary)
    
    # Visualize
    fig = go.Figure()
    
    # Add range bars
    for tenor in rate_summary.index:
        data = rate_summary.loc[tenor]
        fig.add_trace(go.Scatter(
            x=[tenor, tenor],
            y=[data['Low'], data['High']],
            mode='lines',
            line=dict(width=8, color='lightblue'),
            showlegend=False
        ))
        fig.add_trace(go.Scatter(
            x=[tenor],
            y=[data['Last']],
            mode='markers',
            marker=dict(size=12, color='darkblue'),
            name='Last Trade',
            showlegend=tenor == rate_summary.index[0]
        ))
    
    fig.update_layout(
        title='Rate Levels by Benchmark Tenor (Range: Low-High, Marker: Last)',
        xaxis_title='Tenor',
        yaxis_title='Rate (%)',
        height=400
    )
    fig.show()

display_rate_monitor(df)

## 5. Volume Tracker

In [ ]:
def display_volume_tracker(df: pd.DataFrame):
    """
    Display cumulative volume throughout the session.
    """
    if df.empty:
        print("No data for volume tracking.")
        return
    
    # Sort by time
    vol_df = df.sort_values('Event_Time_NY').copy()
    vol_df['Cumulative_Trades'] = range(1, len(vol_df) + 1)
    vol_df['Cumulative_Notional'] = vol_df['Notional amount-Leg 1'].cumsum()
    
    # 15-minute buckets
    vol_df['Time_Bucket'] = vol_df['Event_Time_NY'].dt.floor('15min')
    bucket_summary = vol_df.groupby('Time_Bucket').agg({
        'Dissemination Identifier': 'count',
        'Notional amount-Leg 1': 'sum'
    }).reset_index()
    bucket_summary.columns = ['Time', 'Trades', 'Notional']
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Trade Count (15-min buckets)', 'Notional ($M, 15-min buckets)'])
    
    fig.add_trace(
        go.Bar(x=bucket_summary['Time'], y=bucket_summary['Trades'], marker_color='steelblue'),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Bar(x=bucket_summary['Time'], y=bucket_summary['Notional']/1e6, marker_color='darkgreen'),
        row=2, col=1
    )
    
    fig.update_layout(title_text='Session Volume Tracker', height=500, showlegend=False)
    fig.show()
    
    # Also show cumulative
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=vol_df['Event_Time_NY'],
        y=vol_df['Cumulative_Notional']/1e9,
        mode='lines',
        fill='tozeroy',
        fillcolor='rgba(70, 130, 180, 0.3)',
        line=dict(color='steelblue', width=2)
    ))
    
    fig2.update_layout(
        title='Cumulative Notional Through Session ($B)',
        xaxis_title='Time (NY)',
        yaxis_title='Cumulative Notional ($B)',
        height=400
    )
    fig2.show()

display_volume_tracker(df)

## 6. Tenor Activity Breakdown

In [ ]:
def display_tenor_activity(df: pd.DataFrame):
    """
    Display activity breakdown by tenor.
    """
    if df.empty:
        print("No data for tenor analysis.")
        return
    
    def tenor_bucket(years):
        if pd.isna(years) or years <= 0:
            return 'Unknown'
        elif years <= 2:
            return 'Front End (0-2Y)'
        elif years <= 5:
            return 'Belly (2-5Y)'
        elif years <= 10:
            return 'Intermediate (5-10Y)'
        else:
            return 'Long End (10Y+)'
    
    df_copy = df.copy()
    df_copy['Tenor_Bucket'] = df_copy['Tenor_Years'].apply(tenor_bucket)
    
    tenor_summary = df_copy.groupby('Tenor_Bucket').agg({
        'Dissemination Identifier': 'count',
        'Notional amount-Leg 1': 'sum',
        'Rate_Pct': 'mean'
    }).round(4)
    tenor_summary.columns = ['Trade_Count', 'Total_Notional', 'Avg_Rate']
    
    # Reorder
    bucket_order = ['Front End (0-2Y)', 'Belly (2-5Y)', 'Intermediate (5-10Y)', 'Long End (10Y+)', 'Unknown']
    tenor_summary = tenor_summary.reindex([t for t in bucket_order if t in tenor_summary.index])
    
    print(f"\n{'='*60}")
    print("TENOR ACTIVITY BREAKDOWN")
    print(f"{'='*60}")
    display(tenor_summary)
    
    # Pie chart
    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]],
                        subplot_titles=['Trade Count', 'Notional'])
    
    fig.add_trace(
        go.Pie(labels=tenor_summary.index, values=tenor_summary['Trade_Count'],
               textinfo='label+percent', hole=0.4),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Pie(labels=tenor_summary.index, values=tenor_summary['Total_Notional'],
               textinfo='label+percent', hole=0.4),
        row=1, col=2
    )
    
    fig.update_layout(title_text='Session Activity by Tenor', height=400)
    fig.show()

display_tenor_activity(df)

## 7. Refresh Dashboard

In [ ]:
def refresh_dashboard():
    """
    Refresh all dashboard components.
    """
    clear_output(wait=True)
    
    now = datetime.datetime.now(NY_tz)
    today_start = NY_tz.localize(datetime.datetime(now.year, now.month, now.day, 6, 0))
    
    print(f"Last Refresh: {now.strftime('%Y-%m-%d %H:%M:%S')} NY")
    print(f"Fetching data...")
    
    raw_df = sdr.grab_sdr_trades(
        start_timestamp=today_start,
        end_timestamp=now,
        agency="CFTC",
        asset_class="RATES"
    )
    
    df = process_for_display(raw_df)
    
    display_session_stats(df)
    display_recent_trades(df, n=15)
    display_large_trades(df)
    display_rate_monitor(df)
    
    return df

# Manual refresh
# Uncomment the line below to refresh the dashboard
# df = refresh_dashboard()

## 8. Auto-Refresh Mode (Optional)

In [ ]:
def auto_refresh_dashboard(refresh_seconds: int = 60, max_iterations: int = 10):
    """
    Auto-refresh dashboard at specified interval.
    
    WARNING: This will continuously refresh. Use max_iterations to limit.
    """
    for i in range(max_iterations):
        try:
            refresh_dashboard()
            print(f"\n[Auto-refresh {i+1}/{max_iterations}. Next refresh in {refresh_seconds}s. Interrupt kernel to stop.]")
            time.sleep(refresh_seconds)
        except KeyboardInterrupt:
            print("\nAuto-refresh stopped.")
            break

# Uncomment to enable auto-refresh (will run for 10 iterations)
# auto_refresh_dashboard(refresh_seconds=60, max_iterations=10)

## 9. Quick Summary

In [ ]:
print("\n" + "="*70)
print("REAL-TIME MONITORING DASHBOARD - QUICK REFERENCE")
print("="*70)
print("""
Functions available:

1. refresh_dashboard()     - Manually refresh all components
2. display_session_stats() - Current session statistics
3. display_recent_trades() - Latest trades feed
4. display_large_trades()  - Large trade alerts
5. display_rate_monitor()  - Rate levels by tenor
6. display_volume_tracker()- Volume accumulation
7. display_tenor_activity()- Breakdown by tenor

Auto-refresh:
- Uncomment and run auto_refresh_dashboard() for continuous updates
- Use Kernel > Interrupt to stop auto-refresh

Configuration:
- Modify MonitorConfig class for thresholds and settings
""")